In [4]:
# 라이브러리 호출
import pandas as pd
import numpy as np

In [ ]:
# 데이터 로드
df = pd.read_csv('20260530_전처리-일부완료.csv', encoding='utf-8')

df.head(3)

### 파생변수 - 프리미엄 매물 여부 계산
- IQR을 기준으로 상위 이상치인 값들을 프리미엄이라고 정의하고, 
페르소나에 따라 다르게 적용해보려고 했었음

In [ ]:
# 파생변수 4 : 프리미엄 여부 (is_PM) 

# 상위 이상치의 경계값을 구하는 함수
def high_outlier_idx_group(s, weight=1.5):
    q_1 = np.percentile(s.values, 25)
    q_3 = np.percentile(s.values, 75)
    IQR = q_3 - q_1
    high = q_3 + IQR * weight
    
    # high보다 큰 값들의 index 반환
    return s.index[s > high]

In [ ]:
# 자치구명 + 건물용도 기준으로 그룹화해서 프리미엄 index 추출
pm_idx = df.groupby(["CGG_NM", "BLDG_USG"])["THING_AMT"].apply(high_outlier_idx_group)

In [ ]:
# 중첩된 index를 1차원으로 펼치고 정수형으로 변환
pm_idx_2 = pm_idx.explode().astype(int)

In [ ]:
# 원본 데이터프레임에서 해당 index면 True, 아니면 False
df["is_PM"] = df.index.isin(pm_idx_2)

In [ ]:
# 결과 확인
total = len(df)
pm_count = df["is_PM"].sum()
pm_ratio = pm_count / total * 100

print(f"전체 데이터: {total:,}건")
print(f"프리미엄(is_PM=True): {pm_count:,}건")
print(f"프리미엄 비율: {pm_ratio:.2f}%")

In [ ]:
# True/False → 1/0으로 변환 후 저장
df["is_PM"] = df["is_PM"].astype(int)
df.to_csv("prop_copy5.csv", index=False)
print("저장 완료!")